# Demand Forecasting EDA

This notebook explores the Store Item Demand Forecasting dataset before modelling.

The goal is to understand demand behaviour across time, stores, and products, and to motivate the feature engineering choices used in the forecasting pipeline.

## Key Questions

- Is there visible seasonality?
- Do stores behave differently?
- Are products heterogeneous?
- Are lag and rolling-window features likely to be useful?

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

plt.style.use("ggplot")

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_PATH = PROJECT_ROOT / "data" / "raw" / "train.csv"

df = pd.read_csv(DATA_PATH)
df["date"] = pd.to_datetime(df["date"])

df.head()

## Dataset Overview

The dataset contains daily sales observations at store-item level.
Each row represents sales for a specific product in a specific store on a specific date.

In [ ]:
print(f"Rows: {df.shape[0]:,}")
print(f"Columns: {df.shape[1]:,}")
print(f"Date range: {df['date'].min().date()} to {df['date'].max().date()}")
print(f"Stores: {df['store'].nunique()}")
print(f"Items: {df['item'].nunique()}")

df.describe()

## Missing Values

Before modelling, we check whether the dataset contains missing values.

In [ ]:
df.isna().sum()

## Total Daily Demand

This plot shows total sales across all stores and items by day.

A clear long-term and seasonal pattern would support the use of time-based features.

In [ ]:
daily_sales = df.groupby("date")["sales"].sum()

plt.figure(figsize=(12, 6))
plt.plot(daily_sales.index, daily_sales.values)
plt.title("Total Daily Sales")
plt.xlabel("Date")
plt.ylabel("Sales")
plt.tight_layout()
plt.show()

## Weekly Seasonality

Retail demand often varies by day of week.

This section checks whether average sales differ across weekdays.

In [ ]:
df["day_name"] = df["date"].dt.day_name()

weekday_order = [
    "Monday", "Tuesday", "Wednesday", "Thursday",
    "Friday", "Saturday", "Sunday"
]

weekly_sales = (
    df.groupby("day_name")["sales"]
    .mean()
    .reindex(weekday_order)
)

plt.figure(figsize=(10, 5))
plt.bar(weekly_sales.index, weekly_sales.values)
plt.title("Average Sales by Day of Week")
plt.xlabel("Day of Week")
plt.ylabel("Average Sales")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## Monthly Seasonality

This view checks whether sales levels vary by month.

In [ ]:
df["month"] = df["date"].dt.month

monthly_sales = df.groupby("month")["sales"].mean()

plt.figure(figsize=(10, 5))
plt.bar(monthly_sales.index, monthly_sales.values)
plt.title("Average Sales by Month")
plt.xlabel("Month")
plt.ylabel("Average Sales")
plt.tight_layout()
plt.show()

## Store-Level Differences

Different stores may have different demand levels.

This matters because the model should learn store-level demand differences.

In [ ]:
store_sales = df.groupby("store")["sales"].mean()

plt.figure(figsize=(10, 5))
plt.bar(store_sales.index.astype(str), store_sales.values)
plt.title("Average Sales by Store")
plt.xlabel("Store")
plt.ylabel("Average Sales")
plt.tight_layout()
plt.show()

## Product-Level Differences

Products may have very different baseline demand levels.

This supports using item-level identifiers and historical lag features.

In [ ]:
item_sales = df.groupby("item")["sales"].mean()

plt.figure(figsize=(12, 5))
plt.hist(item_sales.values, bins=20)
plt.title("Distribution of Average Sales by Item")
plt.xlabel("Average Item Sales")
plt.ylabel("Number of Items")
plt.tight_layout()
plt.show()

## Example Store-Item Time Series

This section inspects one individual store-item series.

A good forecasting model should capture the local trend and repeated seasonal patterns of each individual series.

In [ ]:
example_store = 1
example_item = 1

example = df[
    (df["store"] == example_store) &
    (df["item"] == example_item)
].copy()

plt.figure(figsize=(12, 5))
plt.plot(example["date"], example["sales"])
plt.title(f"Sales Time Series - Store {example_store}, Item {example_item}")
plt.xlabel("Date")
plt.ylabel("Sales")
plt.tight_layout()
plt.show()

## Rolling Demand Behaviour

Rolling averages help smooth short-term volatility.

This is one reason rolling-window features are useful in the modelling pipeline.

In [ ]:
example["rolling_mean_28"] = example["sales"].rolling(window=28).mean()

plt.figure(figsize=(12, 5))
plt.plot(example["date"], example["sales"], label="Daily Sales")
plt.plot(example["date"], example["rolling_mean_28"], label="28-Day Rolling Mean")
plt.title(f"Sales vs 28-Day Rolling Mean - Store {example_store}, Item {example_item}")
plt.xlabel("Date")
plt.ylabel("Sales")
plt.legend()
plt.tight_layout()
plt.show()

## Key Findings

- Demand has clear temporal structure.
- Weekly and monthly seasonality are visible.
- Stores have different average demand levels.
- Products have heterogeneous demand levels.
- Individual store-item series show volatility but also repeated patterns.
- Lag and rolling-window features are likely to be predictive.

These findings motivate the modelling approach used in the main forecasting pipeline:

- date-based features
- lag features
- rolling mean features
- time-based validation
- baseline benchmarking
- LightGBM regression